# Homework 7.2 - Task 1

## Marcus Passages and Fixed Conditioning Design

This corrected Task 1 reuses the completed Homework 5.1 *Meditations* corpus instead of downloading Marcus Aurelius again.

The source corpus is the same Casaubon translation already cleaned and verified in Homework 5.1.

The target corpus is Samuel Butler's English prose translation of Homer's *Odyssey*.

Because Project Gutenberg may be unavailable, this notebook uses MIT's Internet Classics Archive as the automatic source for the Butler *Odyssey* text.

The five conditions are fixed before generated output is examined:

- discipline;
- mortality;
- anger;
- duty;
- self-command.

The later statistical generator and direct LLM comparison receive only the theme label, not the entire Marcus passage.

## 1. Source files used by this notebook

The *Meditations* file is local and comes directly from Homework 5.1:

`data/hw7_2/meditations_body.txt`

The supplied file contains the same cleaned twelve-book Casaubon corpus used previously.

For Samuel Butler's *Odyssey*, the notebook accepts either of two local inputs:

1. the raw MIT Internet Classics Archive text at `data/hw7_2/raw/odyssey_butler_mit.txt`, from which Books I through XXIV are extracted; or
2. the already-cleaned packaged text at `data/hw7_2/odyssey_butler_body.txt`.

If neither local file exists, the notebook attempts to download the raw MIT text. This makes the submitted package runnable even when internet access is unavailable.

In [14]:
from pathlib import Path
import hashlib
import json
import re
import shutil
import urllib.request

import numpy as np
import pandas as pd

cwd = Path.cwd()
PROJECT_ROOT = cwd.parent if cwd.name.lower() == "codes" else cwd

DATA_DIR = PROJECT_ROOT / "data" / "hw7_2"
RAW_DIR = DATA_DIR / "raw"
RESULT_DIR = PROJECT_ROOT / "result images" / "hw7_2"

DATA_DIR.mkdir(parents=True, exist_ok=True)
RAW_DIR.mkdir(parents=True, exist_ok=True)
RESULT_DIR.mkdir(parents=True, exist_ok=True)

MEDITATIONS_BODY = DATA_DIR / "meditations_body.txt"
MEDITATIONS_METADATA = DATA_DIR / "meditations_hw5_1_metadata.csv"

ODYSSEY_MIT_URL = "https://classics.mit.edu/Homer/odyssey.mb.txt"
ODYSSEY_RAW = RAW_DIR / "odyssey_butler_mit.txt"
ODYSSEY_BODY = DATA_DIR / "odyssey_butler_body.txt"

print("Project root:", PROJECT_ROOT.resolve())
print("HW7.2 data folder:", DATA_DIR.resolve())

Project root: C:\Users\benja\OneDrive\Univeristy of Alabama courses\Math 638\Project 2\HW7.2
HW7.2 data folder: C:\Users\benja\OneDrive\Univeristy of Alabama courses\Math 638\Project 2\HW7.2\data\hw7_2


## 2. Verify or recover the Homework 5.1 Meditations corpus

The package supplied with this notebook already includes `meditations_body.txt`.

The code also supports a few common Homework 5.1 folder layouts. If the supplied copy is missing but the original Homework 5.1 folder is still beside this project, the notebook copies `meditations_cleaned.txt` automatically.

In [15]:
if not MEDITATIONS_BODY.exists():
    hw5_candidates = [
        PROJECT_ROOT.parent / "HW5.1" / "data" / "meditations_cleaned.txt",
        PROJECT_ROOT.parent / "HW5_1" / "data" / "meditations_cleaned.txt",
        PROJECT_ROOT / "data" / "hw5_1" / "meditations_cleaned.txt",
        PROJECT_ROOT / "data" / "meditations_cleaned.txt",
    ]

    source = next(
        (path for path in hw5_candidates if path.exists()),
        None,
    )

    if source is None:
        raise FileNotFoundError(
            "The Homework 5.1 Meditations corpus was not found.\n"
            "Place the supplied meditations_body.txt at:\n"
            f"{MEDITATIONS_BODY.resolve()}"
        )

    shutil.copy2(source, MEDITATIONS_BODY)
    print("Copied HW5.1 Meditations corpus from:", source.resolve())

meditations_body = MEDITATIONS_BODY.read_text(
    encoding="utf-8",
    errors="replace",
)

if len(meditations_body) < 250_000:
    raise ValueError(
        "The Meditations file looks too short to be the complete cleaned HW5.1 corpus."
    )

# Hash normalized text rather than platform-specific line endings.
med_normalized_sha256 = hashlib.sha256(
    meditations_body.encode("utf-8")
).hexdigest()

if MEDITATIONS_METADATA.exists():
    hw5_metadata = pd.read_csv(MEDITATIONS_METADATA)
    expected_hash = str(hw5_metadata.iloc[0]["sha256"])
    if med_normalized_sha256 != expected_hash:
        raise ValueError(
            "The supplied Meditations text does not match the normalized Homework 5.1 corpus hash."
        )

print("Using local Meditations corpus:", MEDITATIONS_BODY.resolve())
print("Characters:", f"{len(meditations_body):,}")
print("Normalized SHA-256:", med_normalized_sha256)

Using local Meditations corpus: C:\Users\benja\OneDrive\Univeristy of Alabama courses\Math 638\Project 2\HW7.2\data\hw7_2\meditations_body.txt
Characters: 312,633
Normalized SHA-256: 4317d3565d43d4254a1623f8e961eea8f79418bd0b4d599db0343e8b0f03db73


## 3. Obtain or reuse Samuel Butler's Odyssey

The target translation is Samuel Butler's public-domain English prose translation of Homer's *Odyssey*.

The preferred provenance source is the MIT Internet Classics Archive plain-text edition:

`https://classics.mit.edu/Homer/odyssey.mb.txt`

For reproducibility, the submitted package already contains the cleaned Books I through XXIV text. If that cleaned file exists, no network access is required. If a raw MIT file is available, the notebook verifies it and can rebuild the cleaned body. If neither local file exists, the notebook attempts the MIT download.

In [16]:
def download_odyssey_if_needed():
    if ODYSSEY_RAW.exists():
        print("Using existing raw Odyssey file:", ODYSSEY_RAW.resolve())
        return "raw"

    if ODYSSEY_BODY.exists():
        print("Using packaged cleaned Odyssey body:", ODYSSEY_BODY.resolve())
        return "cleaned"

    request = urllib.request.Request(
        ODYSSEY_MIT_URL,
        headers={
            "User-Agent": "Mozilla/5.0 Math638-HW7.2 educational project",
        },
    )

    try:
        with urllib.request.urlopen(request, timeout=30) as response:
            data = response.read()

        ODYSSEY_RAW.write_bytes(data)
        print("Downloaded Samuel Butler Odyssey from MIT:", ODYSSEY_RAW.resolve())
        return "raw"

    except Exception as exc:
        raise RuntimeError(
            "Neither a local Odyssey corpus nor the MIT download was available.\n\n"
            "Place the cleaned text at:\n"
            f"{ODYSSEY_BODY.resolve()}\n\n"
            "or place the raw MIT text at:\n"
            f"{ODYSSEY_RAW.resolve()}\n\n"
            f"Original download error: {exc}"
        )


ODYSSEY_INPUT_MODE = download_odyssey_if_needed()

Using existing raw Odyssey file: C:\Users\benja\OneDrive\Univeristy of Alabama courses\Math 638\Project 2\HW7.2\data\hw7_2\raw\odyssey_butler_mit.txt


## 4. Verify the Butler Odyssey and isolate the literary body

If the raw MIT file is present, the notebook verifies the title, translator, `BOOK I`, `BOOK XXIV`, and `THE END`, then keeps the literary text from `BOOK I` through immediately before `THE END`.

If only the packaged cleaned body is present, the notebook verifies that it begins with `BOOK I`, contains `BOOK XXIV`, and is long enough to be the complete twenty-four-book text. This offline path uses exactly the cleaned target corpus stored with the submission.

In [17]:
if ODYSSEY_INPUT_MODE == "raw":
    odyssey_raw_text = ODYSSEY_RAW.read_text(
        encoding="utf-8",
        errors="replace",
    )

    required_markers = [
        "The Odyssey",
        "Translated by Samuel Butler",
        "BOOK I",
        "BOOK XXIV",
        "THE END",
    ]

    missing_markers = [
        marker
        for marker in required_markers
        if marker.lower() not in odyssey_raw_text.lower()
    ]

    if missing_markers:
        raise ValueError(
            "The local Odyssey file does not look like the expected Samuel Butler text. "
            f"Missing marker(s): {missing_markers}"
        )

    start_match = re.search(
        r"(?m)^BOOK I\s*$",
        odyssey_raw_text,
        flags=re.IGNORECASE,
    )

    end_match = re.search(
        r"(?m)^THE END\s*$",
        odyssey_raw_text,
        flags=re.IGNORECASE,
    )

    if start_match is None or end_match is None or end_match.start() <= start_match.start():
        raise RuntimeError(
            "Could not isolate Books I-XXIV in the MIT Odyssey text."
        )

    odyssey_body = odyssey_raw_text[
        start_match.start():end_match.start()
    ].strip()

    ODYSSEY_BODY.write_text(
        odyssey_body,
        encoding="utf-8",
    )

else:
    odyssey_body = ODYSSEY_BODY.read_text(
        encoding="utf-8",
        errors="replace",
    ).strip()

    if not re.search(r"(?m)^BOOK I\s*$", odyssey_body, flags=re.IGNORECASE):
        raise ValueError("The packaged Odyssey body does not begin with BOOK I.")

    if not re.search(r"(?m)^BOOK XXIV\s*$", odyssey_body, flags=re.IGNORECASE):
        raise ValueError("The packaged Odyssey body does not contain BOOK XXIV.")

    if len(odyssey_body) < 500_000:
        raise ValueError("The packaged Odyssey body looks too short to contain all 24 books.")


book_headings = re.findall(
    r"(?m)^BOOK [IVXLCDM]+\s*$",
    odyssey_body,
    flags=re.IGNORECASE,
)

if len(book_headings) != 24:
    raise ValueError(
        f"Expected 24 Odyssey book headings, found {len(book_headings)}."
    )

ody_sha256 = hashlib.sha256(
    odyssey_body.encode("utf-8")
).hexdigest()

print("Cleaned Odyssey body:", ODYSSEY_BODY.resolve())
print("Characters:", f"{len(odyssey_body):,}")
print("Normalized SHA-256:", ody_sha256)

Cleaned Odyssey body: C:\Users\benja\OneDrive\Univeristy of Alabama courses\Math 638\Project 2\HW7.2\data\hw7_2\odyssey_butler_body.txt
Characters: 610,469
Normalized SHA-256: 51ea5d35fc8f443f324724eb6cf0c8cdd8e30d984339b30f0402ab6e4c1bdb83


## 5. Save corpus provenance

The source-corpus provenance is recorded before passage selection or generation.

In [18]:
provenance = {
    "meditations": {
        "author": "Marcus Aurelius",
        "translator": "Meric Casaubon",
        "source": "reused directly from completed Homework 5.1",
        "local_file": MEDITATIONS_BODY.relative_to(PROJECT_ROOT).as_posix(),
        "characters": len(meditations_body),
        "normalized_sha256": med_normalized_sha256,
    },
    "odyssey": {
        "author": "Homer",
        "translator": "Samuel Butler",
        "source": "MIT Internet Classics Archive plain-text edition",
        "source_url": ODYSSEY_MIT_URL,
        "input_mode": ODYSSEY_INPUT_MODE,
        "raw_local_file": ODYSSEY_RAW.relative_to(PROJECT_ROOT).as_posix(),
        "cleaned_local_file": ODYSSEY_BODY.relative_to(PROJECT_ROOT).as_posix(),
        "characters": len(odyssey_body),
        "normalized_sha256": ody_sha256,
        "cleaning_rule": "retain BOOK I through immediately before THE END when raw source is available",
    },
}

provenance_path = DATA_DIR / "task1_corpus_provenance.json"
provenance_path.write_text(
    json.dumps(provenance, indent=2),
    encoding="utf-8",
)

print(json.dumps(provenance, indent=2))
print("Saved:", provenance_path.resolve())

{
  "meditations": {
    "author": "Marcus Aurelius",
    "translator": "Meric Casaubon",
    "source": "reused directly from completed Homework 5.1",
    "local_file": "data/hw7_2/meditations_body.txt",
    "characters": 312633,
    "normalized_sha256": "4317d3565d43d4254a1623f8e961eea8f79418bd0b4d599db0343e8b0f03db73"
  },
  "odyssey": {
    "author": "Homer",
    "translator": "Samuel Butler",
    "source": "MIT Internet Classics Archive plain-text edition",
    "source_url": "https://classics.mit.edu/Homer/odyssey.mb.txt",
    "input_mode": "raw",
    "raw_local_file": "data/hw7_2/raw/odyssey_butler_mit.txt",
    "cleaned_local_file": "data/hw7_2/odyssey_butler_body.txt",
    "characters": 610469,
    "normalized_sha256": "51ea5d35fc8f443f324724eb6cf0c8cdd8e30d984339b30f0402ab6e4c1bdb83",
    "cleaning_rule": "retain BOOK I through immediately before THE END when raw source is available"
  }
}
Saved: C:\Users\benja\OneDrive\Univeristy of Alabama courses\Math 638\Project 2\HW7.2\dat

## 6. Choose several short Marcus passages reproducibly

The Homework 5.1 cleaned *Meditations* corpus is normalized to lowercase.

Therefore this corrected notebook splits sentences using punctuation only. It does not require the next sentence to begin with a capital letter.

For each theme, the notebook:

1. splits the complete cleaned corpus into candidate sentences;
2. keeps sentences between 12 and 55 words;
3. scores each sentence using a fixed theme search list;
4. selects the highest-scoring unused sentence;
5. breaks ties by earlier corpus position.

The full Marcus passage is stored for interpretation. Later generation receives only the short theme label.

In [19]:
THEME_ORDER = [
    "discipline",
    "mortality",
    "anger",
    "duty",
    "self-command",
]

MARCUS_SEARCH_WORDS = {
    "discipline": [
        "work",
        "labour",
        "practice",
        "action",
        "nature",
        "reason",
    ],
    "mortality": [
        "death",
        "die",
        "dead",
        "life",
        "mortal",
        "dissolution",
    ],
    "anger": [
        "anger",
        "angry",
        "offence",
        "injury",
        "wrong",
        "revenge",
    ],
    "duty": [
        "duty",
        "work",
        "common",
        "good",
        "action",
        "office",
    ],
    "self-command": [
        "mind",
        "self",
        "passion",
        "reason",
        "soul",
        "desire",
    ],
}


def normalize_spaces(text):
    return re.sub(r"\s+", " ", text).strip()


def words(text):
    return re.findall(
        r"[A-Za-z]+(?:'[A-Za-z]+)?",
        text.lower(),
    )


meditations_flat = normalize_spaces(
    meditations_body
)

# Correct for the lowercased HW5.1 corpus: punctuation-only split.
candidate_sentences = re.split(
    r"(?<=[.!?])\s+",
    meditations_flat,
)

candidate_records = []

for index, sentence in enumerate(candidate_sentences):
    token_list = words(sentence)

    if 12 <= len(token_list) <= 55:
        candidate_records.append({
            "sentence_index": index,
            "passage": sentence.strip(),
            "tokens": token_list,
        })

selected = []
used_indices = set()

for theme in THEME_ORDER:
    keyword_set = set(
        MARCUS_SEARCH_WORDS[theme]
    )

    scored = []

    for record in candidate_records:
        if record["sentence_index"] in used_indices:
            continue

        score = len(
            keyword_set
            & set(record["tokens"])
        )

        scored.append((
            -score,
            record["sentence_index"],
            record,
        ))

    scored.sort(
        key=lambda item: (item[0], item[1])
    )

    best_negative_score, _, best = scored[0]
    score = -best_negative_score

    if score <= 0:
        raise RuntimeError(
            f"No candidate passage matched the fixed search list for theme {theme}."
        )

    used_indices.add(
        best["sentence_index"]
    )

    selected.append({
        "theme": theme,
        "sentence_index": best["sentence_index"],
        "search_score": score,
        "word_count": len(best["tokens"]),
        "passage": best["passage"],
    })

selected_passages = pd.DataFrame(selected)

passage_path = DATA_DIR / "task1_selected_marcus_passages.csv"
selected_passages.to_csv(
    passage_path,
    index=False,
)

print(
    selected_passages[
        [
            "theme",
            "sentence_index",
            "search_score",
            "word_count",
            "passage",
        ]
    ].to_string(index=False)
)

print()
print("Saved:", passage_path.resolve())

       theme  sentence_index  search_score  word_count                                                                                                                                                                                                                                                                                    passage
  discipline             864             3          16                                                                                                                                                                                            to a reasonable creature, the same action is both according to nature, and according to reason.
   mortality             195             3          35                                                                                                                 the life of such an one, death can never surprise as imperfect; as of an actor, that should die before he had ended, or the play itself were at an end, a man

## 7. Freeze the conditioning word lists before generation

The rewarded words are the only words that receive a direct probability boost later.

The unboosted probe words are recorded separately so later analysis can check whether conditioning changes anything beyond the explicitly rewarded vocabulary.

For the anger theme, the probe emphasizes restraint-related language because a Marcus passage about anger can counsel restraint rather than anger itself.

In [20]:
REWARDED_WORDS_REQUESTED = {
    "discipline": [
        "endure",
        "patient",
        "command",
        "obey",
        "labour",
    ],
    "mortality": [
        "death",
        "die",
        "dead",
        "mortal",
        "grave",
    ],
    "anger": [
        "anger",
        "angry",
        "wrath",
        "rage",
        "fury",
    ],
    "duty": [
        "duty",
        "obey",
        "command",
        "service",
        "oath",
    ],
    "self-command": [
        "master",
        "mind",
        "heart",
        "hold",
        "endure",
    ],
}

UNBOOSTED_PROBES_REQUESTED = {
    "discipline": [
        "steady",
        "rule",
        "habit",
        "control",
        "restraint",
    ],
    "mortality": [
        "life",
        "fate",
        "perish",
        "end",
        "dust",
    ],
    "anger": [
        "calm",
        "peace",
        "restrain",
        "forgive",
        "gentle",
    ],
    "duty": [
        "task",
        "honour",
        "work",
        "responsibility",
        "obligation",
    ],
    "self-command": [
        "control",
        "restraint",
        "govern",
        "temper",
        "steady",
    ],
}

CONDITIONING_STRENGTH = 1.5
MAX_GENERATED_WORDS = 30
DRAWS_PER_THEME_PER_MODE = 500
BASE_GENERATION_SEED = 63872

DISPLAY_RULE = (
    "For each theme and mode, display the smallest sample_index "
    "with 8 to 30 words and not truncated; if none exists, use "
    "the smallest sample_index."
)

design = {
    "themes": THEME_ORDER,
    "marcus_search_words": MARCUS_SEARCH_WORDS,
    "rewarded_words_requested": REWARDED_WORDS_REQUESTED,
    "unboosted_probes_requested": UNBOOSTED_PROBES_REQUESTED,
    "conditioning_strength_lambda": CONDITIONING_STRENGTH,
    "max_generated_words": MAX_GENERATED_WORDS,
    "draws_per_theme_per_mode": DRAWS_PER_THEME_PER_MODE,
    "base_generation_seed": BASE_GENERATION_SEED,
    "display_rule": DISPLAY_RULE,
    "direct_llm_information": "theme label only",
    "meditations_source": "reused Homework 5.1 cleaned corpus",
    "odyssey_source": "MIT Internet Classics Archive, Samuel Butler translation",
}

design_path = DATA_DIR / "task1_fixed_experiment_design.json"
design_path.write_text(
    json.dumps(design, indent=2),
    encoding="utf-8",
)

print(json.dumps(design, indent=2))
print("Saved:", design_path.resolve())

{
  "themes": [
    "discipline",
    "mortality",
    "anger",
    "duty",
    "self-command"
  ],
  "marcus_search_words": {
    "discipline": [
      "work",
      "labour",
      "practice",
      "action",
      "nature",
      "reason"
    ],
    "mortality": [
      "death",
      "die",
      "dead",
      "life",
      "mortal",
      "dissolution"
    ],
    "anger": [
      "anger",
      "angry",
      "offence",
      "injury",
      "wrong",
      "revenge"
    ],
    "duty": [
      "duty",
      "work",
      "common",
      "good",
      "action",
      "office"
    ],
    "self-command": [
      "mind",
      "self",
      "passion",
      "reason",
      "soul",
      "desire"
    ]
  },
  "rewarded_words_requested": {
    "discipline": [
      "endure",
      "patient",
      "command",
      "obey",
      "labour"
    ],
    "mortality": [
      "death",
      "die",
      "dead",
      "mortal",
      "grave"
    ],
    "anger": [
      "anger",
      "angry",
   

# Task 1 Outputs

After this notebook runs successfully, Task 2 has everything it needs:

- `data/hw7_2/meditations_body.txt`
- `data/hw7_2/odyssey_butler_body.txt`
- `data/hw7_2/task1_corpus_provenance.json`
- `data/hw7_2/task1_selected_marcus_passages.csv`
- `data/hw7_2/task1_fixed_experiment_design.json`

The raw Butler text is also cached at:

`data/hw7_2/raw/odyssey_butler_mit.txt`

No Project Gutenberg request is made anywhere in this corrected Task 1.